# NSE 全上場銘柄 データ取得パイプライン

National Stock Exchange of India (NSE) の全上場銘柄データを取得し、
SQLite データベース (`data/cache/nse/nse_index.db`) に格納する 4 段階パイプライン。

## 概要

| Phase   | 内容                                    | 所要時間 |
| ------- | --------------------------------------- | -------- |
| Phase 1 | 全上場株マスタ（EQUITY_L.csv）          | 〜1 分   |
| Phase 2 | インデックス構成 + sector/industry 補完 | 〜5 分   |
| Phase 3 | 株主構成マスタ（2,200+ 銘柄ループ）     | 〜20 分  |
| Phase 4 | XBRL 詳細株主データ                     | 〜10 分  |

**合計所要時間: 30〜40 分（全銘柄実行時）**

## 出力テーブル

- `stocks` — 全上場銘柄マスタ（symbol, company_name, isin, sector, industry, ...）
- `index_members` — インデックス構成銘柄（NIFTY 50 等 80+ インデックス）
- `shareholdings` — 四半期別株主構成（promoter_pct, public_pct, ...）
- `shareholding_detail` — XBRL 詳細株主データ（機関投資家別）

## 前提条件

```bash
# 依存パッケージが入っていること
uv sync --all-extras
```


In [1]:
# Cell 2: Imports
from __future__ import annotations

import sqlite3
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

from market.nse import (
    IndicesCollector,
    NseSession,
    ShareholdingCollector,
    StockListCollector,
)

print("Imports OK")


Imports OK


In [ ]:
# Cell 3: Config
# ---------------------------------------------------------------------------
# 設定: 必要に応じて変更してください
# ---------------------------------------------------------------------------

# データベースパス（既存 DB と互換スキーマ）
# AIDEV-TODO QC-08: 大規模データ処理時は DuckDB への移行を検討
#   （SQLite の write ロック競合を解消、集計クエリも高速化）
DB_PATH: str = "data/cache/nse/nse_index.db"

# CSV エクスポート先
EXPORT_DIR: str = "data/exports/nse"

# Phase スキップフラグ（既に取得済みの Phase をスキップする場合に True へ）
SKIP_PHASE_1: bool = False  # 全上場株マスタ
SKIP_PHASE_2: bool = False  # インデックス構成
SKIP_PHASE_3: bool = False  # 株主構成マスタ
SKIP_PHASE_4: bool = False  # XBRL 詳細

# デバッグ用: 銘柄数を制限（0 = 全銘柄）
LIMIT_SYMBOLS: int = 0  # 例: 10 で動作確認、0 で全銘柄

# Phase 3 の対象ユニバース（index_members からこのインデックスに属する銘柄のみ取得）
# 既定: NIFTY TOTAL MKT (750 銘柄、NIFTY Smallcap 250 / Microcap 250 を完全包含)。
# 空リストにすると stocks テーブル全件（約 2,794 銘柄、24h コース）にフォールバック。
PHASE3_UNIVERSE_INDICES: list[str] = ["NIFTY TOTAL MKT"]

# Phase 3 の並列取得 worker 数。NSE API はレートリミットがあるため過度な並列化は
# cookie invalidation / 429 を誘発する。2=安全 / 3=推奨 / 4=攻め。
PHASE3_MAX_WORKERS: int = 3

# Phase 4 の並列取得 worker 数。XBRL は archives ホスト (nsearchives.nseindia.com)
# にある静的ファイルで API より制限が緩く、3-5 並列まで安定する傾向。
PHASE4_MAX_WORKERS: int = 3

# Phase 4（XBRL 詳細）の対象絞り込み: Phase 3 で取得した最新 as_on_date の
# promoter_pct がこの値を超える銘柄のみ XBRL を取得する。
PROMOTER_PCT_MIN_THRESHOLD: float = 10.0

# ポライト遅延（秒）
INDEX_DELAY_SEC: float = 0.3
# AIDEV-NOTE: SHAREHOLDING_DELAY_SEC は Phase 3 では未使用（NseSession 側の
# polite_delay に一本化）。Phase 4 の XBRL 取得では XBRL_DELAY_SEC を使用する。
SHAREHOLDING_DELAY_SEC: float = 0.5
XBRL_DELAY_SEC: float = 0.3

# Phase 3/4 スキーマ検証: 割合フィールドの許容範囲（%）
_PCT_MIN: float = 0.0
_PCT_MAX: float = 100.0

# executemany() バッチサイズ（commit 間隔）
_BATCH_COMMIT_SIZE: int = 500

# ---------------------------------------------------------------------------
# ユーティリティ関数
# ---------------------------------------------------------------------------

def _now_iso() -> str:
    """現在の UTC タイムスタンプを ISO 8601 形式で返す。"""
    return datetime.now(timezone.utc).isoformat()


# AIDEV-NOTE: _safe_float / _safe_int は market.nse.parsers.clean_price /
# clean_volume と機能が近いが、挙動が微妙に異なる（_MISSING_VALUES の
# センチネル扱い・NaN/inf 安全性等）ため、Notebook がスタンドアロンで
# pandas DataFrame の雑多な値を扱えるよう残置している。
# DB 挿入前の最終変換に限定して使用すること。


def _safe_float(value: object) -> float | None:
    """値を float に変換し、失敗時は None を返す（Notebook 内専用）。"""
    if value is None or value == "":
        return None
    try:
        return float(value)  # type: ignore[arg-type]
    except (ValueError, TypeError):
        return None


def _safe_int(value: object) -> int | None:
    """値を int に変換し、失敗時は None を返す（Notebook 内専用）。"""
    if value is None or value == "":
        return None
    try:
        return int(value)  # type: ignore[arg-type]
    except (ValueError, TypeError):
        return None


def _in_pct_range(value: float | None) -> bool:
    """割合値が [0, 100] の範囲内または None であることを判定する。"""
    if value is None:
        return True
    return _PCT_MIN <= value <= _PCT_MAX


print(f"DB_PATH      = {DB_PATH}")
print(f"EXPORT_DIR   = {EXPORT_DIR}")
print(f"LIMIT_SYMBOLS= {LIMIT_SYMBOLS} ({'全銘柄' if LIMIT_SYMBOLS == 0 else f'{LIMIT_SYMBOLS} 銘柄のみ'})")
print(f"PHASE3_UNIVERSE_INDICES = {PHASE3_UNIVERSE_INDICES or '[] → stocks 全件'}")
print(f"PHASE3_MAX_WORKERS = {PHASE3_MAX_WORKERS} (並列 worker 数)")
print(f"PHASE4_MAX_WORKERS = {PHASE4_MAX_WORKERS} (並列 worker 数)")
print(f"PROMOTER_PCT_MIN_THRESHOLD = {PROMOTER_PCT_MIN_THRESHOLD}%  (Phase 4 対象絞り込み)")
print(f"Skip flags   = Phase1={SKIP_PHASE_1}, Phase2={SKIP_PHASE_2}, "
      f"Phase3={SKIP_PHASE_3}, Phase4={SKIP_PHASE_4}")


DB_PATH      = data/cache/nse/nse_index.db
EXPORT_DIR   = data/exports/nse
LIMIT_SYMBOLS= 0 (全銘柄)
PHASE3_UNIVERSE_INDICES = ['NIFTY TOTAL MKT']
PHASE3_MAX_WORKERS = 3 (並列 worker 数)
PHASE4_MAX_WORKERS = 3 (並列 worker 数)
PROMOTER_PCT_MIN_THRESHOLD = 10.0%  (Phase 4 対象絞り込み)
Skip flags   = Phase1=False, Phase2=False, Phase3=False, Phase4=False


## Phase 1: 全上場株マスタ

`StockListCollector.fetch_stock_list()` で EQUITY_L.csv を取得し、
`stocks` テーブルに INSERT します。

- 対象: NSE に上場している全株式（約 2,263 銘柄）
- 取得内容: symbol, company_name, isin, series, date_of_listing, face_value
- 冪等: `INSERT OR REPLACE` で再実行時は上書き


In [3]:
# Cell 5: Phase 1 — 全上場株マスタ (EQUITY_L.csv → stocks テーブル)

# DDL（既存 nse_index.db スキーマと完全互換）
_CREATE_STOCKS_DDL = """
CREATE TABLE IF NOT EXISTS stocks (
    symbol          TEXT PRIMARY KEY,
    company_name    TEXT NOT NULL,
    isin            TEXT,
    series          TEXT DEFAULT 'EQ',
    listing_date    TEXT,
    face_value      REAL,
    industry        TEXT,
    sector          TEXT,
    basic_industry  TEXT,
    macro           TEXT,
    is_fno          INTEGER,
    last_price      REAL,
    previous_close  REAL,
    year_high       REAL,
    year_low        REAL,
    ffmc            REAL,
    pct_change_30d  REAL,
    pct_change_365d REAL,
    fetched_at      TEXT NOT NULL
)
"""

_CREATE_INDEX_MEMBERS_DDL = """
CREATE TABLE IF NOT EXISTS index_members (
    index_name  TEXT NOT NULL,
    symbol      TEXT NOT NULL,
    priority    INTEGER,
    fetched_at  TEXT NOT NULL,
    PRIMARY KEY (index_name, symbol),
    FOREIGN KEY (symbol) REFERENCES stocks(symbol)
)
"""

_CREATE_SHAREHOLDINGS_DDL = """
CREATE TABLE IF NOT EXISTS shareholdings (
    symbol              TEXT NOT NULL,
    as_on_date          TEXT NOT NULL,
    promoter_pct        REAL,
    public_pct          REAL,
    employee_trust_pct  REAL DEFAULT 0,
    submission_date     TEXT,
    broadcast_date      TEXT,
    xbrl_url            TEXT,
    fetched_at          TEXT NOT NULL,
    PRIMARY KEY (symbol, as_on_date),
    FOREIGN KEY (symbol) REFERENCES stocks(symbol)
)
"""

_CREATE_SHAREHOLDING_DETAIL_DDL = """
CREATE TABLE IF NOT EXISTS shareholding_detail (
    symbol              TEXT NOT NULL,
    report_date         TEXT NOT NULL,
    category            TEXT NOT NULL,
    sub_category        TEXT NOT NULL DEFAULT '',
    shareholder_name    TEXT NOT NULL DEFAULT '',
    pan                 TEXT NOT NULL DEFAULT '',
    num_shareholders    INTEGER,
    num_fully_paid_shares INTEGER,
    num_voting_rights   INTEGER,
    pct_total_shares    REAL,
    pct_fully_diluted   REAL,
    num_shares_demat    INTEGER,
    is_category_total   INTEGER DEFAULT 1,
    fetched_at          TEXT NOT NULL,
    PRIMARY KEY (symbol, report_date, category, sub_category, shareholder_name),
    FOREIGN KEY (symbol) REFERENCES stocks(symbol)
)
"""

# DB 初期化
Path(DB_PATH).parent.mkdir(parents=True, exist_ok=True)
conn = sqlite3.connect(DB_PATH, timeout=30)
conn.execute("PRAGMA journal_mode=WAL")
conn.execute("PRAGMA busy_timeout=30000")
conn.execute("PRAGMA foreign_keys=ON")
conn.execute(_CREATE_STOCKS_DDL)
conn.execute(_CREATE_INDEX_MEMBERS_DDL)
conn.execute(_CREATE_SHAREHOLDINGS_DDL)
conn.execute(_CREATE_SHAREHOLDING_DETAIL_DDL)
conn.commit()
print(f"DB initialized: {DB_PATH}")

# Phase 1 実行
if SKIP_PHASE_1:
    print("[Phase 1] SKIP_PHASE_1=True — スキップ")
else:
    print("[Phase 1] 全上場株マスタ取得中...")
    _stock_collector = StockListCollector()
    _stocks_df = _stock_collector.fetch_stock_list()

    # LIMIT_SYMBOLS 適用
    if LIMIT_SYMBOLS > 0:
        _stocks_df = _stocks_df.head(LIMIT_SYMBOLS)
        print(f"  LIMIT_SYMBOLS={LIMIT_SYMBOLS} — {len(_stocks_df)} 銘柄に制限")

    _insert_sql = """
    INSERT OR REPLACE INTO stocks
        (symbol, company_name, isin, series, listing_date, face_value, fetched_at)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """
    _now = _now_iso()

    # iterrows() ではなく itertuples()+executemany() でバッチ INSERT
    _rows: list[tuple[object, ...]] = []
    for _, row in _stocks_df.iterrows():
        symbol = row.get("symbol", "")
        if not symbol:
            continue
        _rows.append(
            (
                symbol,
                row.get("company_name", ""),
                row.get("isin", ""),
                row.get("series", "EQ"),
                row.get("date_of_listing", ""),
                _safe_float(row.get("face_value")),
                _now,
            )
        )

    conn.executemany(_insert_sql, _rows)
    conn.commit()
    print(f"[Phase 1] 完了: {len(_rows)} 銘柄を stocks テーブルに INSERT")


DB initialized: data/cache/nse/nse_index.db
[Phase 1] 全上場株マスタ取得中...
[Phase 1] 完了: 2280 銘柄を stocks テーブルに INSERT


## Phase 2: インデックス構成 + sector/industry 補完

`IndicesCollector.fetch_all_indices()` で全インデックス一覧を取得し、
各インデックスの構成銘柄を `index_members` テーブルに INSERT します。

同時に `equity-stockIndices` の詳細情報（`industry`, `is_fno`, 株価等）を
`stocks` テーブルの各銘柄に UPDATE します。

- 対象: 全インデックス（NIFTY 50, NIFTY BANK, NIFTY IT 等 80+ インデックス）
- 取得内容: index_members への INSERT + stocks.industry UPDATE
- スキップ対象: BHARATBOND 系（債券）、NIFTY GS 系（国債）


In [4]:
# Cell 7: Phase 2 — インデックス構成 + sector/industry 補完

# 債券・国債インデックスはスキップ
_SKIP_INDEX_PREFIXES = ("BHARATBOND", "NIFTY GS")

if SKIP_PHASE_2:
    print("[Phase 2] SKIP_PHASE_2=True — スキップ")
else:
    print("[Phase 2] 全インデックス一覧取得中...")
    _indices_collector = IndicesCollector()
    _all_indices_df = _indices_collector.fetch_all_indices()

    # equity インデックスのみ抽出
    _index_names = [
        name
        for name in _all_indices_df["index_symbol"].tolist()
        if isinstance(name, str)
        and not any(name.startswith(pfx) for pfx in _SKIP_INDEX_PREFIXES)
    ]
    print(f"[Phase 2] {len(_index_names)} インデックスを処理")

    _stock_insert_sql = (
        "INSERT OR IGNORE INTO stocks (symbol, company_name, fetched_at) "
        "VALUES (?, ?, ?)"
    )
    _member_insert_sql = """
    INSERT OR REPLACE INTO index_members
        (index_name, symbol, priority, fetched_at)
    VALUES (?, ?, ?, ?)
    """
    _stock_update_sql = """
    UPDATE stocks SET
        industry = COALESCE(?, industry),
        is_fno   = COALESCE(?, is_fno),
        last_price      = ?,
        previous_close  = ?,
        year_high       = ?,
        year_low        = ?,
        ffmc            = ?,
        pct_change_30d  = ?,
        pct_change_365d = ?,
        fetched_at      = ?
    WHERE symbol = ?
    """

    _now = _now_iso()
    _total_members = 0
    _total_updates = 0
    _failed_indices: list[str] = []

    for _idx_name in tqdm(_index_names, desc="インデックス処理"):
        try:
            _df = _indices_collector.fetch_index(_idx_name)
            if _df.empty:
                time.sleep(INDEX_DELAY_SEC)
                continue

            # インデックス単位で executemany() バッチを組み立てる
            _stock_upsert_batch: list[tuple[object, ...]] = []
            _member_batch: list[tuple[object, ...]] = []
            _stock_update_batch: list[tuple[object, ...]] = []

            for _, item in _df.iterrows():
                _sym = item.get("symbol", "")
                if not _sym or _sym == _idx_name:
                    continue

                _stock_upsert_batch.append((_sym, "", _now))
                _member_batch.append((_idx_name, _sym, None, _now))
                _stock_update_batch.append(
                    (
                        item.get("series", None),
                        None,  # is_fno
                        _safe_float(item.get("last_price")),
                        _safe_float(item.get("prev_close")),
                        _safe_float(item.get("year_high")),
                        _safe_float(item.get("year_low")),
                        None,  # ffmc
                        None,  # pct_change_30d
                        None,  # pct_change_365d
                        _now,
                        _sym,
                    )
                )

            if _stock_upsert_batch:
                conn.executemany(_stock_insert_sql, _stock_upsert_batch)
            if _member_batch:
                conn.executemany(_member_insert_sql, _member_batch)
                _total_members += len(_member_batch)
            if _stock_update_batch:
                conn.executemany(_stock_update_sql, _stock_update_batch)
                _total_updates += len(_stock_update_batch)

            conn.commit()

        except Exception as e:
            print(f"  [WARNING] {_idx_name}: {e}")
            _failed_indices.append(_idx_name)

        time.sleep(INDEX_DELAY_SEC)

    print(f"[Phase 2] 完了: {_total_members} メンバー行, {_total_updates} 銘柄更新, "
          f"{len(_failed_indices)} 失敗")
    if _failed_indices:
        print(f"  失敗インデックス: {_failed_indices[:10]}")


[Phase 2] 全インデックス一覧取得中...
[Phase 2] 124 インデックスを処理


インデックス処理:   0%|          | 0/124 [00:00<?, ?it/s]

  [WARNING] NIFTY50 USD: Missing 'data' key in index constituents response
[Phase 2] 完了: 9153 メンバー行, 9153 銘柄更新, 1 失敗
  失敗インデックス: ['NIFTY50 USD']


## Phase 3: 株主構成マスタ

`ShareholdingCollector.fetch_shareholding(symbol)` で各銘柄の株主構成を取得し、
`shareholdings` テーブルに INSERT します。

- 対象: `stocks` テーブルの全銘柄（約 2,263 件）
- 取得内容: promoter_pct, public_pct, employee_trust_pct, xbrl_url
- `CorporateShareHolding.to_float_promoter_group_pct()` で REAL 型変換
- 冪等: `INSERT OR REPLACE` で再実行時は上書き
- 各銘柄間に 0.5 秒のポライト遅延


In [ ]:
# Cell 9: Phase 3 — 株主構成マスタ（並列取得）

if SKIP_PHASE_3:
    print("[Phase 3] SKIP_PHASE_3=True — スキップ")
else:
    # AIDEV-NOTE: 対象ユニバースの決定
    # PHASE3_UNIVERSE_INDICES に指定されたインデックスのメンバー銘柄のみを取得。
    # NIFTY TOTAL MKT (750) は NIFTY Smallcap 250 / Microcap 250 を完全包含。
    if PHASE3_UNIVERSE_INDICES:
        _placeholders = ",".join("?" * len(PHASE3_UNIVERSE_INDICES))
        _universe_query = f"""
        SELECT DISTINCT im.symbol
        FROM index_members im
        JOIN stocks s ON s.symbol = im.symbol
        WHERE im.index_name IN ({_placeholders})
        ORDER BY im.symbol
        """  # noqa: S608
        _universe_rows = conn.execute(
            _universe_query, PHASE3_UNIVERSE_INDICES
        ).fetchall()
        _all_symbols = [row[0] for row in _universe_rows]
        print(
            f"[Phase 3] ユニバース {PHASE3_UNIVERSE_INDICES} から "
            f"{len(_all_symbols)} 銘柄を抽出"
        )
        if not _all_symbols:
            print(
                "  [WARNING] ユニバース抽出がゼロ件。index_members が未投入の "
                "可能性があります。Phase 2 を先に実行するか、"
                "PHASE3_UNIVERSE_INDICES=[] にして全件取得してください。"
            )
    else:
        _cursor = conn.execute("SELECT symbol FROM stocks ORDER BY symbol")
        _all_symbols = [row[0] for row in _cursor.fetchall()]
        print(
            f"[Phase 3] PHASE3_UNIVERSE_INDICES=[] のため "
            f"stocks 全件 {len(_all_symbols)} 銘柄を対象に実行（24h コース）"
        )

    if LIMIT_SYMBOLS > 0:
        _all_symbols = _all_symbols[:LIMIT_SYMBOLS]
        print(f"  LIMIT_SYMBOLS={LIMIT_SYMBOLS} — {len(_all_symbols)} 銘柄に制限")

    print(
        f"[Phase 3] {len(_all_symbols)} 銘柄を "
        f"{PHASE3_MAX_WORKERS} 並列で取得中..."
    )

    # AIDEV-NOTE: ThreadPoolExecutor + ThreadLocal NseSession
    # - 各 worker スレッドが独立した NseSession / ShareholdingCollector を保持
    # - NseSession は組み込み polite_delay(0.5s) を持つためスレッド単位で律速される
    # - SQLite の書き込みは main thread で集約（複数 worker からの同時書き込み回避）
    # - fetch_shareholding は HTTP I/O 律速なのでスレッド並列で効果あり
    _tl = threading.local()

    def _get_thread_collector() -> ShareholdingCollector:
        """スレッドローカルな ShareholdingCollector を取得（初回のみ生成）。"""
        collector = getattr(_tl, "collector", None)
        if collector is None:
            _tl.session = NseSession()
            collector = ShareholdingCollector(session=_tl.session)
            _tl.collector = collector
        return collector

    def _fetch_one(sym: str):
        """1 銘柄分の shareholding を取得。例外はタプルに載せて返す。"""
        try:
            collector = _get_thread_collector()
            return sym, collector.fetch_shareholding(sym), None
        except Exception as e:  # noqa: BLE001
            return sym, None, e

    _insert_sql = """
    INSERT OR REPLACE INTO shareholdings
        (symbol, as_on_date, promoter_pct, public_pct,
         employee_trust_pct, submission_date, broadcast_date,
         xbrl_url, fetched_at)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
    _now = _now_iso()
    _total_rows = 0
    _total_skipped_range = 0
    _failed_symbols: list[str] = []
    _pending_batch: list[tuple[object, ...]] = []
    _thread_sessions: list[NseSession] = []

    _t_start = time.monotonic()

    with ThreadPoolExecutor(
        max_workers=PHASE3_MAX_WORKERS,
        thread_name_prefix="nse-sh",
    ) as executor:
        _futures = {executor.submit(_fetch_one, s): s for s in _all_symbols}

        for _future in tqdm(
            as_completed(_futures),
            total=len(_futures),
            desc="株主構成取得（並列）",
        ):
            _sym, _holdings, _err = _future.result()

            if _err is not None:
                _failed_symbols.append(_sym)
                _err_msg = str(_err).lower()
                if "invalid characters" in _err_msg or "must not be empty" in _err_msg:
                    pass  # 想定内（バリデーションエラー）
                else:
                    print(f"  [WARNING] {_sym}: {_err}")
                continue

            for _h in _holdings:
                # AIDEV-NOTE: NSE API は稀に pct を percent×100 (6729 = 67.29%) や
                # ratio (0.6729) 形式で返す。to_normalized_pcts() で sum ベースの
                # 自動検出を行い percentage 形式へ揃える。
                _promoter_pct, _public_pct, _trust_pct, _fmt = _h.to_normalized_pcts()

                if not (
                    _in_pct_range(_promoter_pct)
                    and _in_pct_range(_public_pct)
                    and _in_pct_range(_trust_pct)
                ):
                    print(
                        f"  [WARNING] {_sym} {_h.as_on_date}: 割合値が範囲外 "
                        f"(promoter={_promoter_pct}, public={_public_pct}, "
                        f"trust={_trust_pct}, fmt={_fmt}) — SKIP"
                    )
                    _total_skipped_range += 1
                    continue

                # 自動補正が走ったケースを INFO ログへ出力
                if _fmt in ("x100", "ratio"):
                    print(
                        f"  [INFO] {_sym} {_h.as_on_date}: pct を {_fmt} 形式から "
                        f"自動補正 (promoter={_promoter_pct:.2f}, "
                        f"public={_public_pct:.2f})"
                    )

                _pending_batch.append(
                    (
                        _h.symbol,
                        _h.as_on_date,
                        _promoter_pct,
                        _public_pct,
                        _trust_pct,
                        _h.submission_date or None,
                        _h.broadcast_date or None,
                        _h.xbrl_url or None,
                        _now,
                    )
                )

            # main thread で batch commit（SQLite write lock 競合を回避）
            if len(_pending_batch) >= _BATCH_COMMIT_SIZE:
                conn.executemany(_insert_sql, _pending_batch)
                conn.commit()
                _total_rows += len(_pending_batch)
                _pending_batch = []

    # 残りを flush
    if _pending_batch:
        conn.executemany(_insert_sql, _pending_batch)
        conn.commit()
        _total_rows += len(_pending_batch)

    _elapsed = time.monotonic() - _t_start
    print(
        f"[Phase 3] 完了: {_total_rows} 行を shareholdings に INSERT "
        f"(所要 {_elapsed / 60:.1f} 分)"
    )
    if _total_skipped_range:
        print(f"  範囲外スキップ: {_total_skipped_range} 行")
    if _failed_symbols:
        print(f"  失敗銘柄数: {len(_failed_symbols)} 件（最初の 10 件: {_failed_symbols[:10]}）")


## Phase 4: XBRL 詳細株主データ

`shareholdings.xbrl_url` に格納された XBRL ファイルを取得し、
`shareholding_detail` テーブルに INSERT します。

- 対象: `shareholdings` テーブルの `xbrl_url` が NULL でない最新行
- 取得内容: 機関投資家別（Mutual Fund, FII, Promoter 個人等）詳細データ
- `ShareholdingCollector.fetch_xbrl_detail(url)` で XBRL パース
- 冪等: `WHERE NOT EXISTS` で重複 INSERT を防止


In [ ]:
# Cell 11: Phase 4 — XBRL 詳細株主データ（並列取得 + レジューム対応）

if SKIP_PHASE_4:
    print("[Phase 4] SKIP_PHASE_4=True — スキップ")
else:
    # AIDEV-NOTE: 対象銘柄の絞り込み + レジューム + 日付ソート修正
    # - shareholdings.as_on_date は "DD-MMM-YYYY" 形式の TEXT であり、
    #   単純な MAX() はテキスト比較で "31-MAR-2025" > "31-DEC-2025" になる
    #   （M > D）。YYYY-MM-DD 形式に変換した iso_date を構築し、
    #   その MAX を chronological latest として使用する。
    # - shareholding_detail に既に登録済みの銘柄は除外（レジューム）
    #   → 中断からの再実行が残件のみで済む
    _xbrl_query = """
    WITH sh_iso AS (
        SELECT
            sh.*,
            substr(sh.as_on_date, -4) || '-'
            || CASE substr(sh.as_on_date, 4, 3)
                   WHEN 'JAN' THEN '01' WHEN 'FEB' THEN '02' WHEN 'MAR' THEN '03'
                   WHEN 'APR' THEN '04' WHEN 'MAY' THEN '05' WHEN 'JUN' THEN '06'
                   WHEN 'JUL' THEN '07' WHEN 'AUG' THEN '08' WHEN 'SEP' THEN '09'
                   WHEN 'OCT' THEN '10' WHEN 'NOV' THEN '11' WHEN 'DEC' THEN '12'
                   ELSE '00'
               END
            || '-' || substr(sh.as_on_date, 1, 2) AS iso_date
        FROM shareholdings sh
    ),
    latest AS (
        SELECT symbol, MAX(iso_date) AS max_iso FROM sh_iso GROUP BY symbol
    )
    SELECT s.symbol, s.xbrl_url
    FROM sh_iso s
    JOIN latest l ON l.symbol = s.symbol AND l.max_iso = s.iso_date
    LEFT JOIN (SELECT DISTINCT symbol FROM shareholding_detail) done
        ON done.symbol = s.symbol
    WHERE s.xbrl_url IS NOT NULL
      AND s.xbrl_url != ''
      AND s.promoter_pct > ?
      AND done.symbol IS NULL
    ORDER BY s.symbol
    """
    _xbrl_rows = conn.execute(
        _xbrl_query, (PROMOTER_PCT_MIN_THRESHOLD,)
    ).fetchall()

    if LIMIT_SYMBOLS > 0:
        _xbrl_rows = _xbrl_rows[:LIMIT_SYMBOLS]
        print(f"  LIMIT_SYMBOLS={LIMIT_SYMBOLS} — {len(_xbrl_rows)} 銘柄に制限")

    _already_done = conn.execute(
        "SELECT COUNT(DISTINCT symbol) FROM shareholding_detail"
    ).fetchone()[0]
    print(
        f"[Phase 4] promoter_pct > {PROMOTER_PCT_MIN_THRESHOLD}% の "
        f"残 {len(_xbrl_rows)} 銘柄を {PHASE4_MAX_WORKERS} 並列で取得中... "
        f"(既完了: {_already_done} 銘柄)"
    )

    if not _xbrl_rows:
        print("[Phase 4] 取得対象なし（全件完了済み or 対象ゼロ）— スキップ")
    else:
        # AIDEV-NOTE: ThreadPoolExecutor + ThreadLocal NseSession
        # XBRL は nsearchives.nseindia.com の静的 XML ファイル。
        # API より制限が緩いため 3-5 並列まで安定。
        # DB 書き込みは main thread で集約し、_BATCH_COMMIT_SIZE 単位で commit。
        _tl4 = threading.local()

        def _get_thread_xbrl_collector() -> ShareholdingCollector:
            collector = getattr(_tl4, "collector", None)
            if collector is None:
                _tl4.session = NseSession()
                collector = ShareholdingCollector(session=_tl4.session)
                _tl4.collector = collector
            return collector

        def _fetch_xbrl_one(sym: str, url: str):
            try:
                collector = _get_thread_xbrl_collector()
                return sym, url, collector.fetch_xbrl_detail(url), None
            except Exception as e:  # noqa: BLE001
                return sym, url, None, e

        _xbrl_insert_sql = """
        INSERT OR IGNORE INTO shareholding_detail
            (symbol, report_date, category, sub_category, shareholder_name, pan,
             num_shareholders, num_fully_paid_shares, num_voting_rights,
             pct_total_shares, pct_fully_diluted, num_shares_demat,
             is_category_total, fetched_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """

        _total_xbrl_rows = 0
        _total_skipped_range = 0
        _failed_xbrl: list[str] = []
        _pending_batch: list[tuple[object, ...]] = []
        _now = _now_iso()
        _symbols_since_commit = 0

        _t_start = time.monotonic()

        with ThreadPoolExecutor(
            max_workers=PHASE4_MAX_WORKERS,
            thread_name_prefix="nse-xbrl",
        ) as executor:
            _futures = {
                executor.submit(_fetch_xbrl_one, s, u): (s, u)
                for s, u in _xbrl_rows
            }

            for _future in tqdm(
                as_completed(_futures),
                total=len(_futures),
                desc="XBRL 取得（並列）",
            ):
                _sym, _url, _result, _err = _future.result()

                if _err is not None:
                    print(f"  [WARNING] {_sym} ({_url[:60]}...): {_err}")
                    _failed_xbrl.append(_sym)
                    continue

                for _row in _result.rows:
                    _pct_total = _safe_float(_row.pct_total_shares)
                    _pct_diluted = _safe_float(_row.pct_fully_diluted)

                    if not (
                        _in_pct_range(_pct_total)
                        and _in_pct_range(_pct_diluted)
                    ):
                        _total_skipped_range += 1
                        continue

                    _pending_batch.append(
                        (
                            # AIDEV-NOTE: FK 整合のため stocks に確実に存在するクエリ
                            # 由来の NSE symbol (_sym) を使用する。_result.symbol は
                            # XBRL 内部の Symbol タグ値で、BSE 形式や空文字の場合があり
                            # shareholding_detail の FK 制約に違反し得る。
                            _sym,
                            _result.as_on_date,
                            _row.category,
                            _row.sub_category,
                            _row.shareholder_name,
                            _row.pan,
                            _safe_int(_row.num_shareholders),
                            _safe_int(_row.num_fully_paid_shares),
                            _safe_int(_row.num_voting_rights),
                            _pct_total,
                            _pct_diluted,
                            _safe_int(_row.num_shares_demat),
                            1 if _row.is_category_total == "true" else 0,
                            _now,
                        )
                    )

                _symbols_since_commit += 1

                # N 銘柄ごと or バッチサイズ到達時にまとめて commit
                if (
                    len(_pending_batch) >= _BATCH_COMMIT_SIZE
                    or _symbols_since_commit >= 20
                ):
                    if _pending_batch:
                        conn.executemany(_xbrl_insert_sql, _pending_batch)
                        _total_xbrl_rows += len(_pending_batch)
                        _pending_batch = []
                    conn.commit()
                    _symbols_since_commit = 0

        # 残りを flush
        if _pending_batch:
            conn.executemany(_xbrl_insert_sql, _pending_batch)
            _total_xbrl_rows += len(_pending_batch)
        conn.commit()

        _elapsed = time.monotonic() - _t_start
        print(
            f"[Phase 4] 完了: {_total_xbrl_rows} 行を shareholding_detail に INSERT "
            f"(所要 {_elapsed / 60:.1f} 分)"
        )
        if _total_skipped_range:
            print(f"  範囲外スキップ: {_total_skipped_range} 行")
        if _failed_xbrl:
            print(f"  失敗銘柄数: {len(_failed_xbrl)} 件（最初の 10 件: {_failed_xbrl[:10]}）")


## データ確認・基本集計

各テーブルの件数と代表銘柄（RELIANCE 等）のサンプルデータを表示します。


In [ ]:
# Cell 13: データ確認・基本集計

print("=" * 60)
print("データ確認")
print("=" * 60)

# 各テーブルの件数
for _tbl in ["stocks", "index_members", "shareholdings", "shareholding_detail"]:
    try:
        _cnt = conn.execute(f"SELECT COUNT(*) FROM {_tbl}").fetchone()[0]  # noqa: S608
        print(f"  {_tbl:<25}: {_cnt:>8,} 件")
    except Exception as e:
        print(f"  {_tbl:<25}: ERROR — {e}")

print()

# RELIANCE のサンプル
_sample_symbol = "RELIANCE"

print(f"--- {_sample_symbol}: stocks テーブル ---")
_stocks_sample = pd.read_sql_query(
    "SELECT symbol, company_name, isin, sector, industry, is_fno, last_price FROM stocks WHERE symbol=?",
    conn,
    params=(_sample_symbol,),
)
if not _stocks_sample.empty:
    display(_stocks_sample)  # noqa: F821
else:
    print(f"  {_sample_symbol} が stocks テーブルに存在しません")

print(f"\n--- {_sample_symbol}: shareholdings テーブル（最新 3 件）---")
_sh_sample = pd.read_sql_query(
    """
    SELECT symbol, as_on_date, promoter_pct, public_pct, employee_trust_pct, xbrl_url
    FROM shareholdings
    WHERE symbol=?
    ORDER BY as_on_date DESC
    LIMIT 3
    """,
    conn,
    params=(_sample_symbol,),
)
if not _sh_sample.empty:
    display(_sh_sample)  # noqa: F821
else:
    print(f"  {_sample_symbol} のデータが shareholdings テーブルに存在しません")

print(f"\n--- {_sample_symbol}: shareholding_detail テーブル（先頭 5 件）---")
_detail_sample = pd.read_sql_query(
    """
    SELECT symbol, report_date, category, sub_category, pct_total_shares
    FROM shareholding_detail
    WHERE symbol=? AND is_category_total=1
    ORDER BY report_date DESC, category
    LIMIT 5
    """,
    conn,
    params=(_sample_symbol,),
)
if not _detail_sample.empty:
    display(_detail_sample)  # noqa: F821
else:
    print(f"  {_sample_symbol} のデータが shareholding_detail テーブルに存在しません")

print("\n--- インデックス別銘柄数（上位 10）---")
_idx_counts = pd.read_sql_query(
    """
    SELECT index_name, COUNT(*) AS member_count
    FROM index_members
    GROUP BY index_name
    ORDER BY member_count DESC
    LIMIT 10
    """,
    conn,
)
display(_idx_counts)  # noqa: F821

print("\n--- 高プロモーター保有率 上位 10 銘柄（最新四半期）---")
_promoter_top = pd.read_sql_query(
    """
    SELECT s.symbol, s.company_name, sh.as_on_date, sh.promoter_pct, sh.public_pct
    FROM shareholdings sh
    JOIN stocks s ON s.symbol = sh.symbol
    WHERE sh.promoter_pct IS NOT NULL
      AND sh.as_on_date = (
          SELECT MAX(as_on_date) FROM shareholdings WHERE symbol = sh.symbol
      )
    ORDER BY sh.promoter_pct DESC
    LIMIT 10
    """,
    conn,
)
display(_promoter_top)  # noqa: F821


## CSV エクスポート

各テーブルを `data/exports/nse/` 以下の CSV ファイルに出力します。


In [ ]:
# Cell 15: CSV エクスポート

_export_dir = Path(EXPORT_DIR)
_export_dir.mkdir(parents=True, exist_ok=True)

_export_tables = [
    "stocks",
    "index_members",
    "shareholdings",
    "shareholding_detail",
]

for _tbl in _export_tables:
    try:
        _df_export = pd.read_sql_query(
            f"SELECT * FROM {_tbl}",  # noqa: S608
            conn,
        )
        _out_path = _export_dir / f"{_tbl}.csv"
        _df_export.to_csv(_out_path, index=False, encoding="utf-8-sig")
        print(f"  {_tbl:<25}: {len(_df_export):>8,} 行 → {_out_path}")
    except Exception as e:
        print(f"  {_tbl:<25}: ERROR — {e}")

# DB コネクションクローズ
conn.close()
print("\nDB connection closed.")
print(f"エクスポート完了: {_export_dir.resolve()}")
